# detrflow — RT-DETR Fine-tuning on COCO 2017

[![GitHub](https://img.shields.io/badge/GitHub-detrflow-black?logo=github)](https://github.com/umutonuryasar/detrflow)
[![HF Model](https://img.shields.io/badge/HuggingFace-rtdetr--r50vd--coco-orange?logo=huggingface)](https://huggingface.co/umutonuryasar/rtdetr-r50vd-coco-detrflow)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python)](https://www.python.org/)

Fine-tune **RT-DETR R50** (`PekingU/rtdetr_r50vd`) on **COCO 2017** using HuggingFace Transformers.  
Targets **A100 40 GB** with BF16 mixed precision and gradient accumulation (effective batch = 64).

---

### Before you start
1. `Runtime → Change runtime type → A100 GPU`
2. Run all cells top to bottom
3. Checkpoints are saved to Google Drive after every epoch

---

### Table of Contents
- [0 — GPU Check](#0)
- [1 — Mount Google Drive](#1)
- [2 — Clone Repo & Install Dependencies](#2)
- [3 — Download COCO 2017](#3)
- [4 — Training Config](#4)
- [5 — Smoke Test](#5)
- [6 — Training](#6)
- [6b — Resume from Checkpoint](#6b)
- [7 — COCO mAP Evaluation](#7)
- [8 — Push to Hugging Face Hub](#8)
- [9 — Results & Visualisation](#9)
- [10 — Experiment Log](#10)

## 0 — GPU Check

> **VRAM required:** 40 GB (A100)  
> Verify the runtime is set to A100 before continuing.

In [ ]:
# Verify GPU availability and print hardware info
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (!!)'}")
if torch.cuda.is_available():
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

assert torch.cuda.is_available(), 'No GPU found — go to Runtime > Change runtime type > GPU.'

### Hugging Face Token

Set your token below. Required to push the final model to the Hub (section 8) and avoids model-download rate limits.  
Get one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
# Set Hugging Face token — needed for Hub push and to avoid download rate limits
import os
os.environ['HF_TOKEN'] = 'your_token_here'

## 1 — Mount Google Drive

> **Est. time:** < 1 min  
> Checkpoints (~2 GB each) are written here after every epoch.

In [ ]:
# Mount Google Drive and create the checkpoint directory
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/detrflow'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')

## 2 — Clone Repo & Install Dependencies

> **Est. time:** 2–3 min  
> Installs `transformers`, `pycocotools`, `torchvision`, and other requirements.

In [ ]:
# Clone the detrflow repository (or pull latest if already present)
import os

if not os.path.exists('/content/detrflow'):
    !git clone https://github.com/umutonuryasar/detrflow.git /content/detrflow
else:
    !git -C /content/detrflow pull
    print('Repo already exists — pulled latest.')

%cd /content/detrflow

In [ ]:
# Install all Python dependencies listed in requirements.txt
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 3 — Download COCO 2017

> **Est. time:** ~10 min (A100 instances have fast networking)  
> **Disk:** ~25 GB total. Skip this cell if COCO is already extracted.

In [ ]:
# Download and extract COCO 2017 train/val images and annotations
import os

COCO_DIR = '/content/detrflow/data/coco'
os.makedirs(COCO_DIR, exist_ok=True)

files = {
    'train2017.zip'                : 'http://images.cocodataset.org/zips/train2017.zip',
    'val2017.zip'                  : 'http://images.cocodataset.org/zips/val2017.zip',
    'annotations_trainval2017.zip' : 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
}

for fname, url in files.items():
    dest = f'{COCO_DIR}/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        !wget -q --show-progress {url} -O {dest}
    else:
        print(f'Already exists: {fname}')

if not os.path.exists(f'{COCO_DIR}/train2017'):
    print('Extracting train2017...')
    !unzip -q {COCO_DIR}/train2017.zip -d {COCO_DIR}
if not os.path.exists(f'{COCO_DIR}/val2017'):
    print('Extracting val2017...')
    !unzip -q {COCO_DIR}/val2017.zip -d {COCO_DIR}
if not os.path.exists(f'{COCO_DIR}/annotations'):
    print('Extracting annotations...')
    !unzip -q {COCO_DIR}/annotations_trainval2017.zip -d {COCO_DIR}

print('COCO ready.')
!ls {COCO_DIR}

## 4 — Training Config

> Loads `configs/rtdetr_r50_coco.yaml` and applies A100-optimised overrides.  
> The result is written to `configs/rtdetr_r50_coco_colab.yaml` for use by later cells.

In [ ]:
# Load base config and override settings for A100 (batch 16, BF16, effective batch 64)
import yaml

with open('configs/rtdetr_r50_coco.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['training']['save_dir']         = CKPT_DIR
cfg['training']['batch_size']       = 16
cfg['training']['grad_accum_steps'] = 4
cfg['training']['epochs']           = 12
cfg['training']['fp16']             = False
cfg['training']['bf16']             = True

with open('configs/rtdetr_r50_coco_colab.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(yaml.dump(cfg, default_flow_style=False))

## 5 — Smoke Test

> **Est. time:** ~30 s (first run downloads model weights ~160 MB)  
> **VRAM:** < 2 GB  
> Confirms the model loads and produces valid detections before committing to full training.

In [ ]:
# Run inference on a sample COCO image to confirm the model and predictor work correctly
import sys
sys.path.insert(0, '/content/detrflow')

from inference.predictor import RTDetrPredictor
from inference.visualizer import draw_detections
from PIL import Image
import requests
from IPython.display import display

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
img = Image.open(requests.get(url, stream=True).raw)

predictor  = RTDetrPredictor(model_id='PekingU/rtdetr_r50vd', confidence_threshold=0.5)
detections = predictor.predict(img)

print(f'Detections: {len(detections)}')
for d in detections[:5]:
    b = d['box']
    print(f"  {d['label']:20s} score={d['score']:.3f}  "
          f"box=[{b['x1']:.0f},{b['y1']:.0f},{b['x2']:.0f},{b['y2']:.0f}]")

display(draw_detections(img, detections))

## 6 — Training

> **Est. time:** ~45 min/epoch × 12 epochs ≈ 9 hrs total on A100  
> **VRAM:** ~28 GB (batch 16, BF16, effective batch 64)  
> A checkpoint is saved to Drive after every epoch to guard against session disconnects.

In [ ]:
# Launch the full training run using the Colab-overridden config
!python scripts/train.py --config configs/rtdetr_r50_coco_colab.yaml

## 6b — Resume from Checkpoint

> Run this cell **instead of section 6** if your Colab session disconnected.  
> Automatically finds the latest `epoch_*` checkpoint on Drive and resumes from there.

In [ ]:
# Find the latest epoch checkpoint and resume training from that point
import os

ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.startswith('epoch_')])

if ckpts:
    latest = f'{CKPT_DIR}/{ckpts[-1]}'
    print(f'Resuming from: {latest}')
    !python scripts/train.py --config configs/rtdetr_r50_coco_colab.yaml --resume {latest}
else:
    print('No checkpoint found. Run section 6 first.')

## 7 — COCO mAP Evaluation

> **Est. time:** ~15 min on val2017 (5 000 images)  
> Reports COCO AP, AP50, and AP75 for the latest checkpoint.

In [ ]:
# Evaluate the latest checkpoint on COCO val2017 and print mAP metrics
import os

ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.startswith('epoch_')])

if not ckpts:
    print('No checkpoint found. Run training first.')
else:
    best = f'{CKPT_DIR}/{ckpts[-1]}'
    print(f'Evaluating: {best}')
    !python scripts/evaluate.py --config configs/rtdetr_r50_coco_colab.yaml --checkpoint {best}

## 8 — Push to Hugging Face Hub

> Uploads the best checkpoint to the Hub as a public model card.  
> Requires `HF_TOKEN` set in section 0.

In [ ]:
# Push the best checkpoint to the Hugging Face Hub
import os
from huggingface_hub import login
from transformers import AutoModelForObjectDetection, AutoImageProcessor

# Re-derive best checkpoint so this cell can be run independently
ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.startswith('epoch_')])
if not ckpts:
    raise RuntimeError('No checkpoint found. Run training first.')
best = f'{CKPT_DIR}/{ckpts[-1]}'

login(token=os.environ.get('HF_TOKEN'))

HF_REPO   = 'umutonuryasar/rtdetr-r50vd-coco-detrflow'
model     = AutoModelForObjectDetection.from_pretrained(best)
processor = AutoImageProcessor.from_pretrained(best)

model.push_to_hub(HF_REPO)
processor.push_to_hub(HF_REPO)
print(f'Model pushed to: https://huggingface.co/{HF_REPO}')

## 9 — Results & Visualisation

> Loads 4 random `val2017` images, runs inference with the fine-tuned model,  
> and displays a 2×2 grid of annotated results.

In [ ]:
# Infer on 4 random val2017 images and display an annotated 2x2 matplotlib grid
import os, random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

VAL_DIR  = '/content/detrflow/data/coco/val2017'
sample   = random.sample([f for f in os.listdir(VAL_DIR) if f.endswith('.jpg')], 4)
vis_pred = RTDetrPredictor(model_id=best, confidence_threshold=0.4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
COLORS = plt.cm.tab20.colors

for ax, fname in zip(axes.flat, sample):
    img  = Image.open(os.path.join(VAL_DIR, fname)).convert('RGB')
    dets = vis_pred.predict(img)
    ax.imshow(img)
    for i, d in enumerate(dets):
        b = d['box']
        x, y = b['x1'], b['y1']
        w, h = b['x2'] - b['x1'], b['y2'] - b['y1']
        color = COLORS[i % len(COLORS)]
        ax.add_patch(patches.Rectangle(
            (x, y), w, h, linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x, y - 4, f"{d['label']} {d['score']:.2f}",
                color=color, fontsize=8, fontweight='bold')
    ax.set_title(fname, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results_grid.png')

## 10 — Experiment Log

Fill in after each evaluation run to track progress across epochs.

| Epoch | Train Loss | Val mAP | AP50 | AP75 | Notes |
|------:|-----------:|--------:|-----:|-----:|-------|
| 1     |            |         |      |      |       |
| 2     |            |         |      |      |       |
| 3     |            |         |      |      |       |
| 4     |            |         |      |      |       |
| 5     |            |         |      |      |       |
| 6     |            |         |      |      |       |
| 7     |            |         |      |      |       |
| 8     |            |         |      |      |       |
| 9     |            |         |      |      |       |
| 10    |            |         |      |      |       |
| 11    |            |         |      |      |       |
| 12    |            |         |      |      |       |